# Week 2 Text Preprocess and Benchmark & Integration
ทำการคลีนข้อความและแบ่งประโยคออกเป็น 3 ส่วน โดยใช้ pythainlp ทำการนับ token
1. สั้น
2. กลาง
3. ยาว 

นำประโยคไปให้โมเดลเสียงและเปรีนบเทียบ latency จากนั้นสร้างฟังก์ชั่น api รอรับ keyword จากฟอนต์

In [1]:
!python --version

Python 3.12.9


## Library

In [2]:
import os
import pandas as pd
import pythainlp 

import gtts 

In [10]:
try:
    import torch
    print("มี torch อยู่ไม่มืดแร้ว" "version:", torch.__version__ if hasattr(torch, '__version__') else "ไม่พบเวอร์ชั่น" )
except ImportError:
    print("อ้ายบ่มีtorch")
    

มี torch อยู่ไม่มืดแร้วversion: 2.9.1


## Data

In [3]:
current_path = os.getcwd()
print("ตอนนี้อยู่ใน:" ,{current_path})

dialogue2 = "Dialogue2.csv"

def dialogue(path):
    df = pd.read_csv(path)
    return df

df = dialogue(dialogue2)
print("โหลดข้อมูลสำเร็จ! พร้อมส่งต่อให้แต่ละโมเดลแล้วค่ะ")
df.head(3)


ตอนนี้อยู่ใน: {'/Users/bam/AIDA-chatbot/backend/voice/dialogue2'}
โหลดข้อมูลสำเร็จ! พร้อมส่งต่อให้แต่ละโมเดลแล้วค่ะ


,ID,Category,Length_Type,Scenario,Keyword,Original Script,TTS Script,Emotion
0,G01,Greeting,Long,ทักทายครั้งแรกแบบ1,"สวัสดี, Hello, หวัดดี, สวัสดีค่ะ, สวัสดีครับ, ...",สวัสดีค่า ไอด้านะคะ ยินดีที่ได้รู้จักไอด้าเป็น...,"สวัสดีค่า, ไอด้านะคะ. ยินดีที่ได้รู้จัก, ไอด้า...",Happy
1,G02,Greeting,Long,ทักทายครั้งแรกแบบ2,"สวัสดี, Hello, หวัดดี, สวัสดีค่ะ, สวัสดีครับ, ...",สวัสดีนะ ยินดีที่ได้รู้จักค่ะ ก่อนอื่นขอแนะนำต...,"สวัสดีนะ, ยินดีที่ได้รู้จักค่ะ. ก่อนอื่นขอแนะน...",Talking
2,G03,Greeting,Short,ทักทายแบบที่3,"สวัสดี, Hello, หวัดดี, สวัสดีค่ะ, สวัสดีครับ, ...",Hello สวัสดีนะคะ ไอด้าเอง ยินดีช่วยเหลือค่ะ สง...,"เฮลโล, สวัสดีนะคะ. ไอด้าเอง, ยินดีช่วยเหลือค่ะ...",Happy


## Text Preprocessing using Pythainlp
1. Count each ID to defining a length type for each sentenses ()
2. To improve context before use voice model (Phoneme/Token Preparation)

explaination: ช่วยลดความลำเอียง (Bias) ของผู้วิจัย และทำให้งานวิจัยนี้สามารถทำซ้ำ (Reproduce)
โมเดลนี้เข้ามาช่วยแยกคำเผื่อที่จะจัดเรียงให้โมเดลเสียงอ่านเข้าใจประโยคหรือคำประสมได้มากขึ้น
1. Short: ทักทายทั่วไปพูดคุยทั่วไปที่ไม่ได้ลงรายละเอียดมาก Low Latency
2. Medium: แทนการตอบคำถามทั่วไปในชีวิตประจำวัน 
3. Long: แทนการอธิบายข้อมูลทางเทคนิคหรือวิชาการที่มีความซับซ้อน

![Response Times](JakobNielsen.png)

In [ ]:
import pandas as pd
import re
from pythainlp import word_tokenize
from pythainlp.corpus import thai_words
from tabulate import tabulate
from pythainlp.util import dict_trie

# # 1. สร้างลิสต์คำศัพท์เฉพาะของคณะเรา
# custom_words = {
#     "บียู ครอคส์", "ไอด้า", "ปัญญาประดิษฐ์", "วิทยาการข้อมูล", 
#     "วิศวกรรมศาสตร์", "ดาต้า ไซน์", "เอไอ", "ไพธอน", "อาทิฟิเชียล"
# }

# # 2. รวมคำศัพท์มาตรฐาน + คำศัพท์เฉพาะของเราเข้าด้วยกัน
# all_words = set(thai_words()) | custom_words
# custom_dict = dict_trie(all_words)

def get_word_details(text):
    if pd.isna(text):
        return 0, ""
    clean_text = re.sub(r'[^0-9a-zA-Z\u0e00-\u0e7f\s]', '', str(text))
    tokens = word_tokenize(clean_text, engine="newmm")
    tokens = [t.strip() for t in tokens if t.strip()]
    return len(tokens), "|".join(tokens)
   
def classify_length(count):
    if count <= 15: return "Short"
    elif count <= 40: return "Medium"
    else: return "Long"

# --- 1. คำนวณค่า ---
df['word_count'], df['token_preview'] = zip(*df['TTS Script'].apply(get_word_details))
df['length_newmm'] = df['word_count'].apply(classify_length)

# --- 2. จัดระเบียบ DF สำหรับการใช้งานจริง ---
final_df = df[['ID', 'Category', 'Scenario', 'Keyword', 'Original Script', 
               'TTS Script', 'Emotion', 'word_count', 'length_newmm', 'token_preview']]

# --- 3. บันทึกลงไฟล์ใหม่ ---
final_df.to_csv('processed_dialogue_v2.csv', index=False, encoding='utf-8-sig')

print("บันทึกไฟล์สำเร็จ: processed_dialogue_v2.csv")
print("-" * 30)


print(tabulate(final_df[['ID', 'length_newmm', 'word_count', 'TTS Script']].head(), 
               headers='keys', tablefmt='fancy_grid', showindex=False))

บันทึกไฟล์สำเร็จ: processed_dialogue_v2.csv
------------------------------
╒══════╤════════════════╤══════════════╤═════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════╕
│ ID   │ length_newmm   │   word_count │ TTS Script                                                                                                                                                  │
╞══════╪════════════════╪══════════════╪═════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════╡
│ G01  │ Medium         │           27 │ สวัสดีค่า, ไอด้านะคะ. ยินดีที่ได้รู้จัก, ไอด้าเป็นผู้ช่วยตอบคำถามสำหรับคณะวิศวกรรมศาสตร์, สาขาปัญญาประดิษฐ์และวิทยาการข้อมูลค่ะ. มีอะไรอยากสอบถามไหมคะ?                                 │
├──────┼────────────────┼──────────────┼─────────────────────────────────────────────────────────────

# TTS
1. การวัดประสิทธิภาพของโมเดล
2. การวัดเวลาของระบบทั้งหมด

latency and real-time-factor

![formola](RTF.png)

## Edge

In [66]:
pip install mutagen

Note: you may need to restart the kernel to use updated packages.


In [6]:
import time
import os
import pandas as pd
import edge_tts
import asyncio
from mutagen.mp3 import MP3
from tabulate import tabulate 

# โหลดข้อมูล
df_v2 = pd.read_csv('processed_dialogue_v2.csv')

output_folder = "output_edge_tts"
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

async def run_benchmark(dataframe, output_path):
    latencies = []
    durations = []
    rtf_values = []
    sec_per_tokens = []
    wpm_values = []
    
    print("🚀 กำลังทดสอบ Edge-TTS บนเครื่อง M4 และคำนวณ RTF...")
    
    for index, row in dataframe.iterrows():
        file_path = os.path.join(output_path, f"{row['ID']}_edge.mp3")
        
        # --- 1. จับเวลา Inference Time (Latency) ---
        start = time.perf_counter()
        
        communicate = edge_tts.Communicate(
            row['TTS Script'], 
            "th-TH-PremwadeeNeural", 
            rate="+6%", pitch="+10Hz"
        )
        await communicate.save(file_path)
        
        end = time.perf_counter()
        latency = end - start
        
        # --- 2. วัดความยาวไฟล์เสียง (Audio Duration) ---
        audio = MP3(file_path)
        duration = audio.info.length # วินาที

        # --- 3. คำนวณหาค่าเฉลี่ยวินาทีต่อหน่วยคำ และหน่วยคำต่อนาที ---
        sec_per_token = duration / row['word_count'] if row['word_count'] > 0 else 0
        wpm = (row['word_count'] / duration) * 60 if duration > 0 else 0

        # --- 4. คำนวณ RTF ---
        rtf = latency / duration if duration > 0 else 0
        
        latencies.append(latency)
        durations.append(duration)
        rtf_values.append(rtf)
        sec_per_tokens.append(sec_per_token) 
        wpm_values.append(wpm)               
        
        print(f" ID: {row['ID']} | Latency: {latency:.3f}s | WPM: {wpm:.2f} | RTF: {rtf:.4f}")

    # --- 5. บันทึกค่าลง DataFrame ให้ครบถ้วน ---
    dataframe['Latency'] = latencies
    dataframe['Audio_Duration'] = durations
    dataframe['Sec_per_Token'] = sec_per_tokens
    dataframe['WPM'] = wpm_values
    dataframe['RTF'] = rtf_values
    
    return dataframe

# --- รันการทดสอบ ---
results_df = await run_benchmark(df_v2, output_folder)

# --- สรุปผลตามกลุ่มความยาว ---
summary = results_df.groupby('length_newmm')[['Latency', 'Audio_Duration', 'WPM', 'RTF']].mean().reset_index()
print("\n สรุปค่าเฉลี่ยตามระดับความยาวประโยค:")
print(tabulate(summary, headers='keys', tablefmt='fancy_grid', showindex=False))

# --- แสดงผลตารางภาพรวม ---
print("\n รายงานผลการทดสอบละเอียด (Inference vs Real-time):")
display_df = results_df[['ID', 'length_newmm', 'word_count', 'Latency', 'Audio_Duration', 'Sec_per_Token', 'WPM', 'RTF']]
print(tabulate(display_df, headers='keys', tablefmt='fancy_grid', showindex=False))

# --- ส่งออกไฟล์ ---
final_export_df = results_df[['ID', 'Category','word_count','token_preview', 'length_newmm', 'Latency', 'Sec_per_Token', 'WPM' , 'Audio_Duration', 'RTF']]
final_export_df.to_csv('edgeTTS_benchmark_report.csv', index=False, encoding='utf-8-sig')

print(f"\n💾 บันทึกผลลงไฟล์ 'edgeTTS_benchmark_report.csv' เรียบร้อยแล้ว!")

🚀 กำลังทดสอบ Edge-TTS บนเครื่อง M4 และคำนวณ RTF...
 ID: G01 | Latency: 0.657s | WPM: 155.89 | RTF: 0.0632
 ID: G02 | Latency: 4.251s | WPM: 159.57 | RTF: 0.3140
 ID: G03 | Latency: 0.610s | WPM: 179.32 | RTF: 0.1072
 ID: B01 | Latency: 1.570s | WPM: 235.85 | RTF: 0.4115
 ID: B02 | Latency: 2.236s | WPM: 172.87 | RTF: 0.4956
 ID: B03 | Latency: 1.320s | WPM: 153.37 | RTF: 0.1687
 ID: K01 | Latency: 2.780s | WPM: 257.94 | RTF: 0.4597
 ID: K02 | Latency: 4.901s | WPM: 154.58 | RTF: 0.4354
 ID: K03 | Latency: 2.677s | WPM: 187.07 | RTF: 0.3795
 ID: K04 | Latency: 1.580s | WPM: 159.57 | RTF: 0.4668
 ID: K05 | Latency: 0.617s | WPM: 185.51 | RTF: 0.0908
 ID: K06 | Latency: 7.871s | WPM: 279.33 | RTF: 0.4580
 ID: K07 | Latency: 6.522s | WPM: 240.79 | RTF: 0.3849
 ID: T01 | Latency: 1.887s | WPM: 184.66 | RTF: 0.4468
 ID: T02 | Latency: 3.138s | WPM: 168.39 | RTF: 0.6776

 สรุปค่าเฉลี่ยตามระดับความยาวประโยค:
╒════════════════╤═══════════╤══════════════════╤═════════╤══════════╕
│ length_newmm 

## Gtts

In [8]:
from gtts import gTTS
import time
import os
import pandas as pd
from mutagen.mp3 import MP3
from tabulate import tabulate 

# 1. โหลดข้อมูล
df_v2 = pd.read_csv('processed_dialogue_v2.csv')

output_folder = "output_gtts"
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# 2. นิยามฟังก์ชัน (ตัด async ออกเพราะ gTTS รันปกติได้เลย)
def run_gtts_benchmark(dataframe, output_path):
    latencies = []
    durations = []
    rtf_values = []
    sec_per_tokens = []
    wpm_values = []
    
    print("🚀 กำลังทดสอบระบบ gTTS และคำนวณ RTF...")
    
    for index, row in dataframe.iterrows():
        file_name = f"{row['ID']}_gTTS.mp3"
        full_path = os.path.join(output_path, file_name)
        
        # --- 1. จับเวลา Latency ---
        start = time.perf_counter()
        
        tts = gTTS(
            text=row['TTS Script'], 
            lang='th', 
            slow=False  
        )
        tts.save(full_path) # gTTS save ตรงๆ ไม่ต้อง await
        
        end = time.perf_counter()
        latency = end - start
        
        # --- 2. วัดความยาวไฟล์เสียง ---
        audio = MP3(full_path)
        duration = audio.info.length 

        # --- 3. คำนวณ Metrics ---
        sec_per_token = duration / row['word_count'] if row['word_count'] > 0 else 0
        wpm = (row['word_count'] / duration) * 60 if duration > 0 else 0

        # --- 4. คำนวณ RTF ---
        rtf = latency / duration if duration > 0 else 0
        
        latencies.append(latency)
        durations.append(duration)
        rtf_values.append(rtf)
        sec_per_tokens.append(sec_per_token) 
        wpm_values.append(wpm)               
        
        print(f" ID: {row['ID']} | Latency: {latency:.3f}s | WPM: {wpm:.2f} | RTF: {rtf:.4f}")

    # บันทึกค่าลง DataFrame
    dataframe['Latency'] = latencies
    dataframe['Audio_Duration'] = durations
    dataframe['Sec_per_Token'] = sec_per_tokens
    dataframe['WPM'] = wpm_values
    dataframe['RTF'] = rtf_values
    
    return dataframe

# --- 3. รันการทดสอบ (เรียกชื่อฟังก์ชันให้ถูก และไม่ต้องใช้ await) ---
results_df = run_gtts_benchmark(df_v2, output_folder)

# --- 4. สรุปผลและแสดงตาราง ---
summary = results_df.groupby('length_newmm')[['Latency', 'Audio_Duration', 'WPM', 'RTF']].mean().reset_index()
print("\n📊 สรุปค่าเฉลี่ยตามระดับความยาวประโยค (gTTS):")
print(tabulate(summary, headers='keys', tablefmt='fancy_grid', showindex=False))

print("\n รายงานผลการทดสอบละเอียด:")
display_df = results_df[['ID', 'length_newmm', 'word_count', 'Latency', 'Audio_Duration', 'Sec_per_Token', 'WPM', 'RTF']]
print(tabulate(display_df, headers='keys', tablefmt='fancy_grid', showindex=False))

# --- 5. ส่งออกไฟล์ ---
final_export_df = results_df[['ID', 'Category','word_count','token_preview', 'length_newmm', 'Latency', 'Sec_per_Token', 'WPM' , 'Audio_Duration', 'RTF']]
final_export_df.to_csv('gTTS_benchmark_report.csv', index=False, encoding='utf-8-sig')

print(f"\n💾 บันทึกผลลงไฟล์ 'gTTS_benchmark_report.csv' เรียบร้อยแล้ว!")

🚀 กำลังทดสอบระบบ gTTS และคำนวณ RTF...
 ID: G01 | Latency: 1.348s | WPM: 90.12 | RTF: 0.0750
 ID: G02 | Latency: 1.610s | WPM: 87.21 | RTF: 0.0650
 ID: G03 | Latency: 0.282s | WPM: 108.70 | RTF: 0.0300
 ID: B01 | Latency: 0.255s | WPM: 153.69 | RTF: 0.0435
 ID: B02 | Latency: 0.325s | WPM: 117.75 | RTF: 0.0491
 ID: B03 | Latency: 0.505s | WPM: 92.25 | RTF: 0.0388
 ID: K01 | Latency: 0.391s | WPM: 142.54 | RTF: 0.0358
 ID: K02 | Latency: 1.017s | WPM: 87.14 | RTF: 0.0509
 ID: K03 | Latency: 0.528s | WPM: 105.97 | RTF: 0.0424
 ID: K04 | Latency: 0.232s | WPM: 111.94 | RTF: 0.0482
 ID: K05 | Latency: 0.515s | WPM: 97.95 | RTF: 0.0400
 ID: K06 | Latency: 1.496s | WPM: 137.08 | RTF: 0.0427
 ID: K07 | Latency: 1.448s | WPM: 139.80 | RTF: 0.0496
 ID: T01 | Latency: 0.218s | WPM: 123.57 | RTF: 0.0345
 ID: T02 | Latency: 0.294s | WPM: 97.31 | RTF: 0.0367

📊 สรุปค่าเฉลี่ยตามระดับความยาวประโยค (gTTS):
╒════════════════╤═══════════╤══════════════════╤═════════╤═══════════╕
│ length_newmm   │   Late

## typhoon-ai/llama3.1-typhoon2-audio-8b-instruct

In [ ]:
# Requirements
pip --version
transformer --version
!pip install fairseq# fairseq required pip==24.0 to install & only worked only on python 3.10
!pip install flash-attn

pip 25.3 from /Users/bam/AIDA-chatbot/.venv/lib/python3.12/site-packages/pip (python 3.12)


NameError: name 'transformer' is not defined